# Analytics Interview Practice (Python + Pandas)
**Duration:** ~90 minutes  
**Goal:** Practice *analytics interview thinking* + *Python coding challenges* in one notebook.

---

## Dummy Case Study: FitPulse (Subscription Fitness App)
FitPulse launched a **new landing page + onboarding flow** to improve subscriptions.

You’re given three tables (generated below):
- **users**: user attributes (signup date, language, country, traffic source, experiment variant)
- **events**: session-level behavior (session date, minutes spent, key events)
- **subs**: subscription outcomes (start date, plan, price, churn date)

You will:
1) define KPIs from scratch,  
2) clean & validate data,  
3) compute funnels, retention, revenue metrics,  
4) run basic inference,  
5) spot anomalies,  
6) answer “why” like an interviewer.

> **Rules:** Prefer clear, correct, readable solutions. Handle missing values and edge cases.

In [1]:
# --- Setup ---
import numpy as np
import pandas as pd

np.random.seed(42)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

## Generate Dummy Data
Run the cell below to create in-memory DataFrames: `users`, `events`, `subs`.

In [2]:
# --- Dummy data generation ---

N_USERS = 3000
start = pd.Timestamp("2025-10-01")
end = pd.Timestamp("2025-12-31")
days = (end - start).days + 1

user_id = np.arange(1, N_USERS + 1)

variant = np.random.choice(["control", "treatment"], size=N_USERS, p=[0.5, 0.5])
language = np.random.choice(["en", "hi", "es", "fr"], size=N_USERS, p=[0.55, 0.25, 0.12, 0.08])
country = np.random.choice(["IN", "US", "UK", "ES", "FR"], size=N_USERS, p=[0.55, 0.2, 0.12, 0.07, 0.06])
traffic_source = np.random.choice(["organic", "paid_search", "social", "referral"], size=N_USERS, p=[0.45, 0.3, 0.2, 0.05])

signup_offset = np.random.randint(0, days, size=N_USERS)
signup_date = start + pd.to_timedelta(signup_offset, unit="D")

users = pd.DataFrame({
    "user_id": user_id,
    "signup_date": signup_date,
    "variant": variant,
    "language": language,
    "country": country,
    "traffic_source": traffic_source,
})

# Events: 1–8 sessions per user
sessions_per_user = np.random.randint(1, 9, size=N_USERS)
session_rows = sessions_per_user.sum()

# Expand user_id
event_user_id = np.repeat(user_id, sessions_per_user)

# Session dates after signup (0–30 days)
days_after = np.random.randint(0, 31, size=session_rows)
session_date = (users.set_index("user_id").loc[event_user_id, "signup_date"].values
                + pd.to_timedelta(days_after, unit="D"))

# Minutes spent: treatment slightly higher on average
base_minutes = np.random.gamma(shape=2.2, scale=3.0, size=session_rows)  # positive skew
lift = np.where(users.set_index("user_id").loc[event_user_id, "variant"].values == "treatment", 1.10, 1.00)
minutes_spent = np.clip(base_minutes * lift, 0.2, None)

# Key events: landing_view, signup, subscribe (binary flags at session level)
# We'll generate subscribe propensity based on variant + source + language
src = users.set_index("user_id").loc[event_user_id, "traffic_source"].values
lang = users.set_index("user_id").loc[event_user_id, "language"].values
var = users.set_index("user_id").loc[event_user_id, "variant"].values

p_landing = 0.95  # almost all sessions are landing views in this simplified dataset
landing_view = (np.random.rand(session_rows) < p_landing).astype(int)

# Signup happens in early sessions; increase with treatment a bit
p_signup = 0.18 + (var == "treatment") * 0.02 + (src == "organic") * 0.01
signup_event = (np.random.rand(session_rows) < np.clip(p_signup, 0, 0.6)).astype(int)

# Subscribe conditional-ish: make it rarer; favor treatment + paid_search + en
p_sub = 0.035 + (var == "treatment") * 0.01 + (src == "paid_search") * 0.008 + (lang == "en") * 0.005
subscribe_event = (np.random.rand(session_rows) < np.clip(p_sub, 0, 0.25)).astype(int)

events = pd.DataFrame({
    "user_id": event_user_id,
    "session_date": pd.to_datetime(session_date),
    "minutes_spent": minutes_spent.round(2),
    "landing_view": landing_view,
    "signup_event": signup_event,
    "subscribe_event": subscribe_event,
})

# Subs: one subscription max per user (derived from whether any session had subscribe_event)
sub_users = (events.groupby("user_id")["subscribe_event"].max().reset_index())
sub_users = sub_users[sub_users["subscribe_event"] == 1]["user_id"].values

plan = np.random.choice(["monthly", "annual"], size=len(sub_users), p=[0.82, 0.18])
price = np.where(plan == "monthly", 9.99, 79.99)

# Start date: first session where subscribe_event==1
first_sub_date = (events[events["subscribe_event"] == 1]
                  .groupby("user_id")["session_date"].min()
                  .reindex(sub_users).values)

# Churn date: only for monthly; some churn in 30–90 days
churn_flag = (plan == "monthly") & (np.random.rand(len(sub_users)) < 0.28)
churn_days = np.random.randint(30, 91, size=len(sub_users))
churn_date = np.where(churn_flag, pd.to_datetime(first_sub_date) + pd.to_timedelta(churn_days, unit="D"), pd.NaT)

subs = pd.DataFrame({
    "user_id": sub_users,
    "sub_start_date": pd.to_datetime(first_sub_date),
    "plan": plan,
    "price": price,
    "churn_date": pd.to_datetime(churn_date),
})

# Introduce a few messy values for realism
# - missing language for some users
mask = np.random.rand(N_USERS) < 0.01
users.loc[mask, "language"] = None

# - minutes_spent as string in a few rows
mask2 = np.random.rand(len(events)) < 0.002
events.loc[mask2, "minutes_spent"] = events.loc[mask2, "minutes_spent"].astype(str)

# Quick peek
display(users.head())
display(events.head())
display(subs.head())

print("Rows:", {"users": len(users), "events": len(events), "subs": len(subs)})

C:\Users\abhis\AppData\Local\Temp\ipykernel_35560\1448349964.py:102: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['19.39' '11.98' '17.4' '3.09' '3.17' '2.15' '3.91' '2.96' '5.07' '5.34'
 '13.24' '5.68' '4.56' '4.15' '2.79' '3.25' '5.71' '2.44' '4.82' '6.0'
 '3.34' '3.94' '16.1' '10.64' '6.01']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  events.loc[mask2, "minutes_spent"] = events.loc[mask2, "minutes_spent"].astype(str)


,user_id,signup_date,variant,language,country,traffic_source
0,1,2025-12-23,control,hi,US,social
1,2,2025-11-08,treatment,hi,IN,paid_search
2,3,2025-12-06,treatment,en,ES,paid_search
3,4,2025-10-05,treatment,hi,US,organic
4,5,2025-12-04,control,hi,IN,paid_search


,user_id,session_date,minutes_spent,landing_view,signup_event,subscribe_event
0,1,2026-01-01,4.29,1,0,0
1,1,2025-12-31,6.43,1,0,0
2,1,2025-12-28,5.15,1,1,1
3,1,2025-12-23,2.39,1,0,0
4,1,2025-12-27,9.87,1,0,0


,user_id,sub_start_date,plan,price,churn_date
0,1,2025-12-28,monthly,9.99,NaT
1,8,2025-12-15,monthly,9.99,2026-02-16
2,12,2025-12-06,monthly,9.99,NaT
3,14,2025-10-27,monthly,9.99,2026-01-20
4,16,2025-12-24,annual,79.99,NaT


Rows: {'users': 3000, 'events': 13711, 'subs': 560}


# Questions (10–12)
Each question has a **short interview prompt** + a **coding task** (where applicable).

> Tip: Answer like you’re in an interview—explain your assumptions and tradeoffs briefly.

## Q1 — Define KPIs from scratch (high-level)
**Prompt:** FitPulse wants to “improve growth and revenue.” That’s vague.  
1) Propose **one North Star metric** (define it precisely).  
2) Propose **3 supporting KPIs** (leading indicators).  
3) For each KPI, write: **formula, grain, and why it matters**.

**Deliverable:** a short markdown answer (bullets are fine).

👉 **Your answer here:**

## Q2 — Data model + grain
1) What is the **grain** of each table (`users`, `events`, `subs`)?  
2) What are the **join keys** and what can go wrong (e.g., duplication)?  
3) List **2 data quality checks** you’d run before computing KPIs.

**Deliverable:** short written answer.

👉 **Your answer here:**

## Q3 — Data cleaning (types + missing values)
Clean and standardize:
- `users.language`: fill missing with `"unknown"`
- `events.minutes_spent`: convert to numeric; invalid -> NaN; then fill NaN with median minutes_spent (overall)
- Ensure `session_date`, `signup_date` are datetime

**Deliverable:** updated `users` and `events` dataframes + show `dtypes`.

In [3]:
# TODO: write your cleaning code here
import pandas as pd
users["language"] = users["language"].fillna("unknown")
events["minutes_spent"] = pd.to_numeric(
    events["minutes_spent"], errors="coerce"
)
events["minutes_spent"].fillna(
    events["minutes_spent"].median(), inplace=True
)
events["session_date"] = pd.to_datetime(
    events["session_date"], errors="coerce"
)
users["signup_date"] = pd.to_datetime(
    users["signup_date"], errors="coerce"
)
# Deliverable
users.dtypes, events.dtypes

(user_id                    int32
 signup_date       datetime64[ns]
 variant                   object
 language                  object
 country                   object
 traffic_source            object
 dtype: object,
 user_id                     int32
 session_date       datetime64[ns]
 minutes_spent             float64
 landing_view                int32
 signup_event                int32
 subscribe_event             int32
 dtype: object)

## Q4 — Feature engineering (analytics-style)
Create these derived columns:
- In `events`: `event_date` = date part of `session_date`
- In `users`: `signup_week` = Monday-start week (e.g., using pandas period or dt.to_period)
- In `events`: `days_since_signup` = (session_date - signup_date) in whole days (join needed)

**Deliverable:** show `events[['user_id','session_date','days_since_signup']].head()`

In [9]:
# TODO: create derived columns
# In events: event_date = date part of session_date
events["event_date"] = events["session_date"].dt.date

# In users: signup_week = Monday-start week (e.g., using pandas period or dt.to_period)
users["signup_week"] = users["signup_date"].dt.to_period("W-SUN").dt.start_time # W-SUN is defining the end weekday and therefore, Monday is start day

# In events: days_since_signup = (session_date - signup_date) in whole days (join needed)
events = events.merge(
    users[["user_id", "signup_date"]],
    on="user_id",
    how="inner"
)

events["days_since_signup"] = (events["session_date"] - events["signup_date"]).dt.days
events[['user_id','session_date','days_since_signup']].head()


,user_id,session_date,days_since_signup
0,1,2026-01-01,9
1,1,2025-12-31,8
2,1,2025-12-28,5
3,1,2025-12-23,0
4,1,2025-12-27,4


In [5]:
events.head()

,user_id,session_date,minutes_spent,landing_view,signup_event,subscribe_event,event_date
0,1,2026-01-01,4.29,1,0,0,2026-01-01
1,1,2025-12-31,6.43,1,0,0,2025-12-31
2,1,2025-12-28,5.15,1,1,1,2025-12-28
3,1,2025-12-23,2.39,1,0,0,2025-12-23
4,1,2025-12-27,9.87,1,0,0,2025-12-27


In [6]:
users.head()

,user_id,signup_date,variant,language,country,traffic_source,signup_week
0,1,2025-12-23,control,hi,US,social,2025-12-22
1,2,2025-11-08,treatment,hi,IN,paid_search,2025-11-03
2,3,2025-12-06,treatment,en,ES,paid_search,2025-12-01
3,4,2025-10-05,treatment,hi,US,organic,2025-09-29
4,5,2025-12-04,control,hi,IN,paid_search,2025-12-01


## Q5 — Funnel conversion by variant
Compute funnel rates by `variant`:
- landing → signup (at user level)
- signup → subscribe (at user level)
- landing → subscribe (at user level)

**Notes:**
- Convert session-level flags into **user-level** “has_event” (max).
- Avoid double-counting users with multiple sessions.

**Deliverable:** a table with columns:
`variant, n_users, landing_rate, signup_rate, subscribe_rate, landing_to_signup, signup_to_subscribe, landing_to_subscribe`

In [16]:
# TODO: compute the funnel table
# TODO: compute the funnel table
# reduce events to user level
user_funnel = (
    events.groupby("user_id", as_index=False)[["landing_view", "signup_event", "subscribe_event"]]
    .max()
)
# add variant from users
user_funnel = user_funnel.merge(
    users[["user_id", "variant"]],
    on="user_id",
    how="left"
)
# aggregate by variant: counts + rates
funnel = (
    user_funnel.groupby("variant")
    .agg(
        users=("user_id", "nunique"),
        landing_users=("landing_view", "sum"),
        signup_users=("signup_event", "sum"),
        subscribe_users=("subscribe_event", "sum"),
    )
    .reset_index()
)

funnel['landing_rate'] = funnel.landing_users / funnel.users
funnel['signup_rate'] = funnel.signup_users / funnel.users
funnel['subscribe_rate'] = funnel.subscribe_users / funnel.users

funnel['landing_signup'] = funnel.signup_users / funnel.landing_users
funnel['signup_subscribe'] = funnel.subscribe_users / funnel.signup_users
funnel['landing_subscribe'] = funnel.subscribe_users / funnel.landing_users

funnel

,variant,users,landing_users,signup_users,subscribe_users,landing_rate,signup_rate,subscribe_rate,landing_signup,signup_subscribe,landing_subscribe
0,control,1485,1473,816,252,0.991919,0.549495,0.169697,0.553971,0.308824,0.171079
1,treatment,1515,1501,879,308,0.990759,0.580198,0.203300,0.585610,0.350398,0.205197


## Q6 — Uplift + interpretation (interview style)
Using your funnel table:
1) Compute **absolute uplift** and **relative uplift** for `landing_to_subscribe` (treatment vs control).  
2) Write 2–3 sentences interpreting whether this is “meaningful” and what you’d check next (e.g., bias, novelty, segment effects).

**Deliverable:** print uplift numbers + short written interpretation.

In [18]:
# TODO: compute uplift
# Q6 — Uplift
control = funnel.loc[funnel["variant"] == "control", "landing_subscribe"].iloc[0]
treatment = funnel.loc[funnel["variant"] == "treatment", "landing_subscribe"].iloc[0]
absolute_uplift = treatment - control
relative_uplift = absolute_uplift / control if control > 0 else None
absolute_uplift, relative_uplift


(0.03411710590767053, 0.19942260715078847)

👉 **Interpretation here:**

## Q7 — Inference: minutes spent
**Prompt:** Compare `minutes_spent` between control and treatment.  
1) Run the test and report: group means, mean difference, p-value.

**Deliverable:** short justification + computed results.

In [ ]:
# TODO: run a t-test (use scipy if available, or implement with stats formulas)


👉 **Justification here:**

---
# Optional “Interview Wrap” (2–3 minutes)
Answer briefly:
1) What’s the **single most important insight** from your analysis?  
2) What decision would you recommend (ship / don’t ship / iterate)?  
3) What are 2 risks or confounders in this analysis?

👉 **Your answer here:**